In [1]:
import lightgbm as lgb
import numpy as np
import optuna
import polars as pl
from sklearn.model_selection import train_test_split
from surrogate_model.metrics import enrichment_factor, spearman_corr, top_k_recall
from surrogate_model.optuna import (
    RECALL_TOP_1_PERCENT,
    RECALL_TOP_5_PERCENT,
    make_objective,
)

In [2]:
FEATURES = "data/sampled.parquet"
LABELS = "data/1L83.1L83:p2rank:3.output.parquet"

RANDOM_SEED = 1000
SAMPLE_SIZE = 25000
PRIMARY_METRIC = RECALL_TOP_5_PERCENT

features = pl.read_parquet(FEATURES)
labels = pl.read_parquet(LABELS)
labels = labels["catalog_id", "affinity_kcal_mol"]

df = features.join(labels, on="catalog_id", how="inner")

if SAMPLE_SIZE < len(df):
    df = df.sample(SAMPLE_SIZE, seed=RANDOM_SEED)

In [3]:
FEATURE_NAMES = [
    "heavy_atom_count",
    "molecular_weight",
    "calculated_partition_coefficient",
    "calculated_distribution_coefficient",
    "topological_polar_surface_area",
    "hydrogen_bond_donors",
    "pka",
]

LABEL_NAME = "affinity_kcal_mol"

x = df.select(FEATURE_NAMES).to_numpy()
x = np.hstack([x, np.array(df["morgan_fingerprint"].to_list())])

y = df[LABEL_NAME].to_numpy()

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=RANDOM_SEED
)

study = optuna.create_study(direction="maximize")

study.optimize(
    make_objective(
        X_train,
        y_train,
        5,
        primary_metric=PRIMARY_METRIC,
        random_seed=RANDOM_SEED,
    ),
    n_trials=30,
    show_progress_bar=True,
)

[I 2026-09-07 21:25:48,286] A new study created in memory with name: no-name-28393cb9-0ab8-4af0-bd11-85f1db3e263d


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-07 21:26:03,476] Trial 0 finished with value: 0.16500000000000004 and parameters: {'num_leaves': 194, 'max_depth': 9, 'learning_rate': 0.0021683742197928504, 'n_estimators': 1899, 'min_child_samples': 81, 'subsample': 0.7306051332884453, 'colsample_bytree': 0.6018641274267473, 'reg_alpha': 0.005704445445589061, 'reg_lambda': 7.400845785026775e-05}. Best is trial 0 with value: 0.16500000000000004.
[I 2026-09-07 21:26:05,085] Trial 1 finished with value: 0.13 and parameters: {'num_leaves': 33, 'max_depth': 11, 'learning_rate': 0.07731096441670812, 'n_estimators': 251, 'min_child_samples': 85, 'subsample': 0.9783656716426571, 'colsample_bytree': 0.6754782966767765, 'reg_alpha': 0.005978620207022246, 'reg_lambda': 8.749627359747647e-05}. Best is trial 0 with value: 0.16500000000000004.
[I 2026-09-07 21:26:13,706] Trial 2 finished with value: 0.18 and parameters: {'num_leaves': 177, 'max_depth': 10, 'learning_rate': 0.001023370070466761, 'n_estimators': 1021, 'min_child_samples':

In [5]:
best_params = study.best_params
best_params.update({"random_state": RANDOM_SEED})

final_model = lgb.LGBMRegressor(**best_params)
final_model.fit(
    X_train,
    y_train,
    eval_X=X_test,
    eval_y=y_test,
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=True)],
)

y_pred = np.asarray(final_model.predict(X_test))

results = {
    "top_1_percent": top_k_recall(y_test, y_pred, 0.01),
    "top_5_percent": top_k_recall(y_test, y_pred, 0.05),
    "top_10_percent": top_k_recall(y_test, y_pred, 0.1),
    "spearman": spearman_corr(y_test, y_pred),
    "enrichment_factor": enrichment_factor(y_test, y_pred, 0.05),
}

results

Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1113]	valid_0's l2: 255.243


{'top_1_percent': 0.2,
 'top_5_percent': 0.184,
 'top_10_percent': 0.282,
 'spearman': 0.29510981761483557,
 'enrichment_factor': 3.6799999999999997}

| dataset size | train metric | trials | top_1_percent | top_5_percent | top_10_percent | spearman | enrichment |
| - | - | - | - | - | - | - | - |
| 10000 | RECALL_TOP_1_PERCENT | 20 | 0.05 | 0.13 | 0.265 | 0.256 | 2.6 |
| 10000 | RECALL_TOP_5_PERCENT | 20 | 0.0 | 0.19 | 0.24 | 0.065| 3.8 |
| 25000 | RECALL_TOP_1_PERCENT | 20 | 0.26 | 0.224 | 0.32 | 0.391 | 4.48 |
| 25000 | RECALL_TOP_5_PERCENT | 20 | 0.22 | 0.22 | 0.308 | 0.427 | 4.40 |
| 50000 | RECALL_TOP_1_PERCENT | 20 | 0.16 | 0.276 | 0.307 | 0.358 | 5.52 |
| 50000 | RECALL_TOP_5_PERCENT | 20 | 0.12 | 0.234 | 0.284 | 0.326 | 4.68 |
| 100000 | RECALL_TOP_1_PERCENT | 20 | 0.09 | 0.174 | 0.262 | 0.277 | 3.48 |
| 100000 | RECALL_TOP_5_PERCENT | 20 |  |  |  |  |  |
